# RecurrentBitNet V2

Implementing the architecture described in `DESIGN.md`:

1. **Encoder Stack**: Non-recurrent parsing layers (~15% of depth)
2. **Reasoning Core**: Recurrent circuit blocks with iteration embeddings + adaptive halting
3. **Decoder Stack**: Non-recurrent output layers (~25% of depth)
4. **Training**: Progressive recurrence curriculum, auxiliary per-step loss, differential LRs

**Run the cells in order.** Toggle `USE_REAL_DATA` in §5 to switch between micro-testing and full FineWeb-Edu training.

## 1. Environment Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import os
import time
from dataclasses import dataclass
from typing import Optional
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} — {props.total_mem / 1e9:.1f} GB VRAM")

## 2. BitLinear (1.58-bit Ternary Quantization)

Weights quantized to {-1, 0, +1} via STE during training.
Full-precision latent weights maintained for gradient updates.

In [ ]:
def ste_round(x: torch.Tensor) -> torch.Tensor:
    """Round with Straight-Through Estimator."""
    return x + (x.round() - x).detach()


def quantize_weights_ternary(w: torch.Tensor):
    """Quantize weights to ternary {-1, 0, +1} with per-tensor scale."""
    scale = w.abs().mean().clamp(min=1e-5)
    w_normalized = w / scale
    w_ternary = ste_round(w_normalized).clamp(-1, 1)
    return w_ternary, scale


def quantize_activations_int8(x: torch.Tensor):
    """Quantize activations to int8 with per-token absmax scaling."""
    Qb = 127
    scale = x.abs().max(dim=-1, keepdim=True).values.clamp(min=1e-5)
    x_int = (x * Qb / scale).round().clamp(-Qb, Qb)
    return x_int, scale


class BitLinear(nn.Linear):
    """
    BitLinear drop-in replacement for nn.Linear.
    Trains in FP32/BF16, quantizes to ternary in forward pass.
    """
    def __init__(self, in_features, out_features, bias=False):
        super().__init__(in_features, out_features, bias=bias)

    @torch.amp.custom_fwd(device_type="cuda")
    def forward(self, x):
        # Quantize weights (ternary with STE)
        w_ternary, w_scale = quantize_weights_ternary(self.weight)
        w_effective = self.weight + (w_ternary * w_scale - self.weight).detach()

        # Quantize activations (int8 with STE)
        x_int, x_scale = quantize_activations_int8(x)
        x_effective = x + (x_int * x_scale / 127.0 - x).detach()

        return x_effective @ w_effective.t()

## 3. Architecture Components

RMSNorm, multi-head self-attention, SwiGLU feed-forward, and the
Encoder/Reasoning/Decoder transformer blocks.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(d_model))

    def forward(self, x):
        x_fp32 = x.float()
        norm = x_fp32.pow(2).mean(dim=-1, keepdim=True).add(self.eps).rsqrt()
        return (x_fp32 * norm).to(x.dtype) * self.weight


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = BitLinear(d_model, 3 * d_model, bias=False)
        self.out_proj = BitLinear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        B, L, D = x.size()
        qkv = self.qkv(x).reshape(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        scale = math.sqrt(self.head_dim)
        attn = (q @ k.transpose(-2, -1)) / scale
        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(attn, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, L, D)
        return self.out_proj(out)


class SwiGLUFFN(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.w1 = BitLinear(d_model, d_ff, bias=False)
        self.w2 = BitLinear(d_ff, d_model, bias=False)
        self.w3 = BitLinear(d_model, d_ff, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads)
        self.norm2 = RMSNorm(d_model)
        self.ffn = SwiGLUFFN(d_model, d_ff)

    def forward(self, x, mask=None):
        x = x + self.attn(self.norm1(x), mask)
        x = x + self.ffn(self.norm2(x))
        return x

## 4. Structural Stacks

- **EncoderStack**: Non-recurrent, parses input format → abstract representation
- **ReasoningCore**: Recurrent blocks × R with iteration embeddings + adaptive halting
- **DecoderStack**: Non-recurrent, abstract representation → output tokens

In [ ]:
class EncoderStack(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.d_ff)
            for _ in range(config.encoder_blocks)
        ])

    def forward(self, x, mask=None):
        for block in self.blocks:
            x = block(x, mask)
        return x


class ReasoningCore(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.d_ff)
            for _ in range(config.reasoning_blocks)
        ])
        # Learned signal per recurrence iteration
        max_R = config.reasoning_recurrence
        self.iteration_embeddings = nn.Parameter(
            torch.randn(max_R, 1, 1, config.d_model) * 0.02
        )
        # Lightweight halt scorer (ACT scaffold)
        self.halt_scorer = nn.Sequential(
            nn.Linear(config.d_model, 1),
            nn.Sigmoid()
        )

    def forward(self, x, mask=None, R=None, recurrence_dropout=0.0):
        if R is None:
            R = self.iteration_embeddings.size(0)

        iter_outputs = []
        for r in range(R):
            # Recurrence dropout during training
            if self.training and recurrence_dropout > 0 and r > 0:
                if torch.rand(1).item() < recurrence_dropout:
                    continue

            # Add iteration embedding
            if r < self.iteration_embeddings.size(0):
                x = x + self.iteration_embeddings[r]

            # Run all reasoning blocks
            for block in self.blocks:
                x = block(x, mask)

            iter_outputs.append(x)

        return x, iter_outputs


class DecoderStack(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.d_ff)
            for _ in range(config.decoder_blocks)
        ])

    def forward(self, x, mask=None):
        for block in self.blocks:
            x = block(x, mask)
        return x

## 5. RecurrentBitNet V2 Model Assembly

In [ ]:
@dataclass
class ModelConfig:
    d_model: int = 768
    n_heads: int = 12
    d_ff: int = 3072
    vocab_size: int = 32000
    max_seq_len: int = 1024
    encoder_blocks: int = 2
    reasoning_blocks: int = 5
    reasoning_recurrence: int = 3
    decoder_blocks: int = 2
    recurrence_dropout: float = 0.1


class RecurrentBitNetV2(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)
        self.encoder = EncoderStack(config)
        self.reasoning_core = ReasoningCore(config)
        self.decoder = DecoderStack(config)
        self.final_norm = RMSNorm(config.d_model)
        self.lm_head = BitLinear(config.d_model, config.vocab_size, bias=False)
        # Weight tying
        self.lm_head.weight = self.token_emb.weight

    def forward(self, idx, targets=None, R=None):
        B, L = idx.size()
        x = self.token_emb(idx)
        mask = torch.tril(torch.ones(L, L, device=x.device)).unsqueeze(0).unsqueeze(0)

        x = self.encoder(x, mask)
        x, iter_outputs = self.reasoning_core(
            x, mask, R=R,
            recurrence_dropout=self.config.recurrence_dropout if self.training else 0.0
        )
        x = self.decoder(x, mask)

        x = self.final_norm(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss, iter_outputs


# --- Instantiate ---
config = ModelConfig()
model = RecurrentBitNetV2(config).to(DEVICE)

# Enable gradient checkpointing for VRAM savings
for module in model.modules():
    if hasattr(module, 'gradient_checkpointing_enable'):
        module.gradient_checkpointing_enable()

num_params = sum(p.numel() for p in model.parameters())
print(f"✅ RecurrentBitNet V2 instantiated")
print(f"   Unique parameters: {num_params:,}")
print(f"   Effective depth: {config.encoder_blocks} + {config.reasoning_blocks}×{config.reasoning_recurrence} + {config.decoder_blocks} = {config.encoder_blocks + config.reasoning_blocks * config.reasoning_recurrence + config.decoder_blocks} layers")
print(f"   Recurrence dropout: {config.recurrence_dropout}")

## 6. Training Configuration

Toggle `USE_REAL_DATA = True` below to switch from dummy micro-testing to
full FineWeb-Edu streaming. All other hyperparameters are set here.

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# ⚙️  TOGGLE THIS TO SWITCH BETWEEN MICRO AND REAL TRAINING
USE_REAL_DATA = False
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Training hyperparameters
TOTAL_STEPS   = 100 if not USE_REAL_DATA else 100_000
BATCH_SIZE    = 4
SEQ_LEN       = config.max_seq_len
LOG_EVERY     = 10 if not USE_REAL_DATA else 100
SAVE_EVERY    = 50 if not USE_REAL_DATA else 5_000
EVAL_EVERY    = 50 if not USE_REAL_DATA else 5_000
MAX_GRAD_NORM = 1.0
AUX_DECAY     = 0.3   # α in the auxiliary loss formula

# Progressive recurrence curriculum: (step_threshold, R)
CURRICULUM = [
    (0, 1),
    (TOTAL_STEPS // 5, 2),
    (TOTAL_STEPS // 2, 3),
] if not USE_REAL_DATA else [
    (0,      1),
    (10_000, 2),
    (30_000, 3),
    (60_000, config.reasoning_recurrence),
]

# Checkpoint directory
CHECKPOINT_DIR = './checkpoints_v2'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f"Mode: {'🌐 Real Data (FineWeb-Edu)' if USE_REAL_DATA else '🧪 Micro Testing (dummy data)'}")
print(f"Total steps: {TOTAL_STEPS:,}")
print(f"Curriculum: {CURRICULUM}")
print(f"Checkpoints → {CHECKPOINT_DIR}")

## 7. Data Pipeline

In [ ]:
if USE_REAL_DATA:
    # — FineWeb-Edu Streaming —
    from datasets import load_dataset
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained('TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T')
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    fineweb_stream = load_dataset(
        'HuggingFaceFW/fineweb-edu', name='sample-10BT',
        split='train', streaming=True
    )
    stream_iter = iter(fineweb_stream)

    def get_batch():
        inputs, targets = [], []
        while len(inputs) < BATCH_SIZE:
            try:
                text = next(stream_iter)['text']
            except StopIteration:
                # Restart stream on exhaustion
                global stream_iter
                stream_iter = iter(fineweb_stream)
                text = next(stream_iter)['text']

            tokens = tokenizer(
                text, truncation=True, max_length=SEQ_LEN + 1,
                return_tensors='pt'
            )['input_ids'][0]
            if len(tokens) < SEQ_LEN + 1:
                pad = torch.full((SEQ_LEN + 1 - len(tokens),), tokenizer.pad_token_id, dtype=torch.long)
                tokens = torch.cat([tokens, pad])
            inputs.append(tokens[:SEQ_LEN])
            targets.append(tokens[1:SEQ_LEN + 1])
        return torch.stack(inputs).to(DEVICE), torch.stack(targets).to(DEVICE)

    print(f"✅ FineWeb-Edu streaming ready (tokenizer vocab: {len(tokenizer):,})")

else:
    # — Dummy micro data —
    def get_batch():
        idx = torch.randint(0, config.vocab_size, (BATCH_SIZE, SEQ_LEN), device=DEVICE)
        tgt = torch.randint(0, config.vocab_size, (BATCH_SIZE, SEQ_LEN), device=DEVICE)
        return idx, tgt
    tokenizer = None
    print("✅ Dummy data generator ready")

## 8. Optimizer with Differential Learning Rates

Per DESIGN.md:
- Encoder/Decoder: standard LR (1e-3)
- Reasoning Core: higher LR (2e-3) — most reused weights
- Embeddings: lower LR (5e-4) — large, slow-moving

In [ ]:
param_groups = [
    {'params': list(model.encoder.parameters()) + list(model.decoder.parameters()),
     'lr': 1e-3, 'name': 'encoder_decoder'},
    {'params': list(model.reasoning_core.parameters()),
     'lr': 2e-3, 'name': 'reasoning_core'},
    {'params': [model.token_emb.weight],
     'lr': 5e-4, 'name': 'embeddings'},
    {'params': [model.final_norm.weight],
     'lr': 1e-3, 'name': 'final_norm'},
]

optimizer = torch.optim.AdamW(param_groups, betas=(0.9, 0.95), weight_decay=0.1)

# Cosine LR schedule with linear warmup
warmup_steps = min(200, TOTAL_STEPS // 10)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = (step - warmup_steps) / max(1, TOTAL_STEPS - warmup_steps)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

# Mixed precision scaler
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE == 'cuda'))

print(f"✅ Optimizer ready — {len(param_groups)} param groups, warmup={warmup_steps}")
for g in param_groups:
    n = sum(p.numel() for p in g['params'])
    print(f"   {g['name']}: {n:,} params @ lr={g['lr']}")

## 9. Progressive Recurrence Training Loop

Core loop with:
- Progressive curriculum (R increases over training)
- Auxiliary per-recurrence loss with exponential decay
- Mixed precision (BF16)
- Gradient clipping
- Periodic checkpointing & logging

In [ ]:
model.train()
loss_history = []
recurrence_history = []
step_times = []
best_loss = float('inf')

print("🚀 Starting training...")
print("=" * 60)

for step in range(1, TOTAL_STEPS + 1):
    t0 = time.time()

    # 1. Update curriculum
    R = 1
    for threshold, depth in reversed(CURRICULUM):
        if step >= threshold:
            R = depth
            break

    # 2. Get batch
    idx, targets = get_batch()

    # 3. Forward pass with mixed precision
    with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda'), dtype=torch.bfloat16):
        logits, base_loss, iter_outputs = model(idx, targets, R=R)

        # 4. Auxiliary recurrence loss
        aux_loss = torch.tensor(0.0, device=DEVICE)
        for r, hidden in enumerate(iter_outputs):
            step_normed = model.final_norm(hidden)
            step_logits = model.lm_head(step_normed)
            step_loss = F.cross_entropy(step_logits.view(-1, step_logits.size(-1)), targets.view(-1))
            decay_weight = AUX_DECAY ** (R - (r + 1))
            aux_loss = aux_loss + decay_weight * step_loss

        total_loss = base_loss + aux_loss

    # 5. Backward + optimizer step
    optimizer.zero_grad()
    scaler.scale(total_loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    # 6. Track metrics
    loss_val = total_loss.item()
    loss_history.append(loss_val)
    recurrence_history.append(R)
    step_times.append(time.time() - t0)

    if loss_val < best_loss:
        best_loss = loss_val

    # 7. Logging
    if step % LOG_EVERY == 0:
        avg_time = sum(step_times[-LOG_EVERY:]) / LOG_EVERY
        avg_loss = sum(loss_history[-LOG_EVERY:]) / LOG_EVERY
        lr = scheduler.get_last_lr()[0]
        print(
            f"Step {step:>6d}/{TOTAL_STEPS} | "
            f"Loss {avg_loss:.4f} | "
            f"R={R} | "
            f"LR {lr:.2e} | "
            f"{avg_time*1000:.0f} ms/step"
        )

    # 8. Checkpoint
    if step % SAVE_EVERY == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'checkpoint_step{step}.pt')
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'config': config,
            'loss_history': loss_history,
            'recurrence_history': recurrence_history,
            'best_loss': best_loss,
        }, ckpt_path)
        print(f"  💾 Checkpoint → {ckpt_path}")

print("=" * 60)
print(f"✅ Training complete! Best loss: {best_loss:.4f}")

## 10. Save Final Model

Saves the full model state dict, config, and training history.
Also exports ternary-quantized weights for efficient inference.

In [ ]:
# Save full training state
final_path = os.path.join(CHECKPOINT_DIR, 'final_model.pt')
torch.save({
    'step': TOTAL_STEPS,
    'model_state_dict': model.state_dict(),
    'config': config,
    'loss_history': loss_history,
    'recurrence_history': recurrence_history,
    'best_loss': best_loss,
}, final_path)
print(f"💾 Final model saved → {final_path}")

# Export ternary weights for inference
print("\n📦 Exporting ternary weights...")
ternary_weights = {}
with torch.no_grad():
    for name, module in model.named_modules():
        if isinstance(module, BitLinear) and module.weight is not model.token_emb.weight:
            w_ternary, w_scale = quantize_weights_ternary(module.weight)
            ternary_weights[name] = {
                'weight_ternary': w_ternary.to(torch.int8).cpu(),
                'weight_scale': w_scale.float().cpu(),
            }

export_path = os.path.join(CHECKPOINT_DIR, 'ternary_export.pt')
torch.save(ternary_weights, export_path)
num_ternary = sum(v['weight_ternary'].numel() for v in ternary_weights.values())
size_mb = num_ternary * 0.25 / 1e6
print(f"📦 Ternary export saved → {export_path}")
print(f"   {num_ternary:,} ternary params → ~{size_mb:.1f} MB at TQ2_0")

## 11. Evaluation

Measures perplexity on held-out data and generates sample text
to qualitatively assess model quality.

In [ ]:
@torch.no_grad()
def evaluate_perplexity(model, num_batches=50):
    """Compute perplexity over num_batches of data."""
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    for i in tqdm(range(num_batches), desc="Evaluating"):
        idx, targets = get_batch()
        with torch.amp.autocast('cuda', enabled=(DEVICE == 'cuda'), dtype=torch.bfloat16):
            logits, _, _ = model(idx, targets, R=config.reasoning_recurrence)

        # Compute cross-entropy, ignoring padding if using real data
        logits_flat = logits.view(-1, logits.size(-1))
        targets_flat = targets.view(-1)

        if tokenizer is not None and hasattr(tokenizer, 'pad_token_id'):
            pad_id = tokenizer.pad_token_id
            mask = targets_flat != pad_id
            loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=pad_id, reduction='sum')
            total_tokens += mask.sum().item()
        else:
            loss = F.cross_entropy(logits_flat, targets_flat, reduction='sum')
            total_tokens += targets_flat.numel()

        total_loss += loss.item()

    model.train()
    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(avg_loss, 100))
    return {'loss': avg_loss, 'perplexity': ppl}


results = evaluate_perplexity(model, num_batches=50 if USE_REAL_DATA else 20)
print(f"\n📊 Evaluation Results:")
print(f"   Loss:       {results['loss']:.4f}")
print(f"   Perplexity: {results['perplexity']:.2f}")

## 12. Training Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Loss curve with smoothing
axes[0].plot(loss_history, alpha=0.2, color='steelblue', label='Raw')
if len(loss_history) > 20:
    window = min(50, len(loss_history) // 4)
    smoothed = [
        sum(loss_history[max(0,i-window):i+1]) / len(loss_history[max(0,i-window):i+1])
        for i in range(len(loss_history))
    ]
    axes[0].plot(smoothed, color='steelblue', linewidth=2, label='Smoothed')
axes[0].set_ylabel("Total Loss (Base + Aux)")
axes[0].set_title("RecurrentBitNet V2 — Training Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Recurrence curriculum
axes[1].step(range(len(recurrence_history)), recurrence_history,
             where='post', color='coral', linewidth=2)
axes[1].set_ylabel("Recurrence Depth (R)")
axes[1].set_xlabel("Training Step")
axes[1].set_title("Progressive Recurrence Curriculum")
axes[1].set_yticks(sorted(set(recurrence_history)))
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CHECKPOINT_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f"📈 Plot saved → {CHECKPOINT_DIR}/training_curves.png")